In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import *
import os

In [0]:
currency_list = spark.sql(
    "SELECT distinct dc_currency_code FROM currency.03_gold_currency.dim_currencies"
).toPandas()["dc_currency_code"].tolist()

print(currency_list)

In [0]:
# Parameters
start_date = '01-01-2022'
#end_date = '02-01-2022'
end_date = datetime.now().strftime('%m-%d-%Y')
top = 100
skip = 0

for currency in currency_list:
    skip = 0
    all_data = []
    while True:

        url = (
            f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
            f"CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
            f"@moeda='{currency}'&@dataInicial='{start_date}'&@dataFinalCotacao='{end_date}'"
            f"&$top={top}&$skip={skip}&format=json&$select=cotacaoCompra,dataHoraCotacao"
        )
        response = requests.get(url)
        data = response.json()['value']
        if not data:
            break
        all_data.extend(data)
        skip += top

    if all_data:
        df = spark.createDataFrame(all_data) \
            .withColumn('currency', lit(currency))

        start_path = datetime.strptime(start_date, "%m-%d-%Y").strftime("%Y%m%d")
        end_path = datetime.strptime(end_date, "%m-%d-%Y").strftime("%Y%m%d")

        path = (
            "/Volumes/currency/landing/currency_analysis" + "/Quotes/New/"
            f"{currency}-"
            f"{start_path}_"
            f"{end_path}"
            f".parquet"
        )
        df.write.mode('overwrite').parquet(path)
        display(df)